In [21]:
# Celda 1 - PB-07 / PB-08 / PB-09: imports y acceso al módulo src
# Objetivo: usar la lógica reusable encapsulada en src/preprocessing.py

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from scipy import sparse

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.preprocessing import (
    TARGET_COL,
    clean_data,
    add_features,
    get_model_columns,
    build_pipeline,
)

In [22]:
# Celda 2 - PB-07 / PB-08 / PB-09: carga del dataset crudo
# Objetivo: partir desde data/raw para ejecutar el flujo integrado del Sprint 2

DATA_PATH = ROOT / "data" / "raw" / "dataset_pucp.csv"

df_raw = pd.read_csv(DATA_PATH)

print("Dataset crudo cargado correctamente.")
print("Dimensiones:", df_raw.shape)
display(df_raw.head())

Dataset crudo cargado correctamente.
Dimensiones: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [23]:
# Celda 3 - PB-07 / PB-08: limpieza base y feature engineering
# Objetivo: aplicar las funciones reutilizables del módulo preprocessing

df_clean = clean_data(df_raw)
df_feat = add_features(df_clean)

print("Dimensiones después de limpieza:", df_clean.shape)
print("Dimensiones después de agregar features:", df_feat.shape)

display(df_feat.head())

Dimensiones después de limpieza: (12205, 18)
Dimensiones después de agregar features: (12205, 20)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue,total_paginas,duracion_total
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0,False,1,0.000000
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0,False,2,64.000000
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0,False,1,0.000000
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,0,False,2,2.666667
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,1,False,10,627.500000


In [24]:
# Celda 4 - PB-08: documentación breve de features nuevas
# Objetivo: dejar evidencia clara de las features seleccionadas para el pipeline

feature_docs = pd.DataFrame([
    {
        "feature": "total_paginas",
        "como_se_calcula": "Administrative + Informational + ProductRelated",
        "por_que_podria_ser_util": "Resume el volumen total de navegación del usuario dentro del sitio."
    },
    {
        "feature": "duracion_total",
        "como_se_calcula": "Administrative_Duration + Informational_Duration + ProductRelated_Duration",
        "por_que_podria_ser_util": "Resume el tiempo total de navegación y puede reflejar mayor interés o profundidad de sesión."
    }
])

display(feature_docs)

,feature,como_se_calcula,por_que_podria_ser_util
0,total_paginas,Administrative + Informational + ProductRelated,Resume el volumen total de navegación del usua...
1,duracion_total,Administrative_Duration + Informational_Durati...,Resume el tiempo total de navegación y puede r...


In [25]:
# Celda 5 - PB-09: definición de X e y y split estratificado
# Objetivo: separar train y test antes de cualquier fit o balanceo

X = df_feat.drop(columns=[TARGET_COL]).copy()
y = df_feat[TARGET_COL].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test :", X_test.shape)
print("Dimensiones de y_train:", y_train.shape)
print("Dimensiones de y_test :", y_test.shape)

Dimensiones de X_train: (9764, 19)
Dimensiones de X_test : (2441, 19)
Dimensiones de y_train: (9764,)
Dimensiones de y_test : (2441,)


In [26]:
# Celda 6 - PB-07 / PB-08 / PB-09: construcción del pipeline reusable
# Objetivo: usar el pipeline definido en src/preprocessing.py

num_cols_model, cat_cols_model, bool_cols_model = get_model_columns()
pipeline_preprocessing = build_pipeline(random_state=42)

print("Columnas numéricas del pipeline:")
print(num_cols_model)

print("\nColumnas categóricas del pipeline:")
print(cat_cols_model)

print("\nColumnas booleanas del pipeline:")
print(bool_cols_model)

print("\nPipeline creado correctamente:")
print(pipeline_preprocessing)

Columnas numéricas del pipeline:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'total_paginas', 'duracion_total']

Columnas categóricas del pipeline:
['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']

Columnas booleanas del pipeline:
['Weekend']

Pipeline creado correctamente:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',


In [27]:
# Celda 7 - PB-07 / PB-08 / PB-09: ejecución del pipeline sobre train
# Objetivo: validar preprocesamiento + balanceo con SMOTE

X_train_resampled, y_train_resampled = pipeline_preprocessing.fit_resample(X_train, y_train)

print("Forma de X_train después del pipeline:", X_train_resampled.shape)
print("Forma de y_train después del pipeline:", y_train_resampled.shape)

print("\nDistribución de clases luego del balanceo:")
display(pd.Series(y_train_resampled).value_counts().to_frame("frecuencia"))
display((pd.Series(y_train_resampled).value_counts(normalize=True) * 100).round(2).to_frame("%"))

Forma de X_train después del pipeline: (16476, 75)
Forma de y_train después del pipeline: (16476,)

Distribución de clases luego del balanceo:


,frecuencia
Revenue,
1,8238
0,8238


,%
Revenue,
1,50.0
0,50.0


In [28]:
# Celda 8 - PB-07 / PB-08 / PB-09: transformación del conjunto de prueba
# Objetivo: aplicar al test solo el preprocesamiento, sin balancearlo

X_test_processed = pipeline_preprocessing.named_steps["preprocessor"].transform(X_test)

print("Forma de X_test después del preprocesamiento:", X_test_processed.shape)

Forma de X_test después del preprocesamiento: (2441, 75)


In [29]:
# Celda 9 - PB-07 / PB-08 / PB-09: generación de DataFrames procesados
# Objetivo: convertir train y test procesados en tablas con nombres de columnas

feature_names = pipeline_preprocessing.named_steps["preprocessor"].get_feature_names_out()

if sparse.issparse(X_train_resampled):
    X_train_array = X_train_resampled.toarray()
else:
    X_train_array = X_train_resampled

if sparse.issparse(X_test_processed):
    X_test_array = X_test_processed.toarray()
else:
    X_test_array = X_test_processed

df_train_processed = pd.DataFrame(X_train_array, columns=feature_names)
df_train_processed["Revenue"] = y_train_resampled

df_test_processed = pd.DataFrame(X_test_array, columns=feature_names)
df_test_processed["Revenue"] = y_test.reset_index(drop=True)

print("Dimensiones train procesado:", df_train_processed.shape)
print("Dimensiones test procesado :", df_test_processed.shape)

Dimensiones train procesado: (16476, 76)
Dimensiones test procesado : (2441, 76)


In [30]:
# Celda 10 - PB-07 / PB-08 / PB-09: guardado de outputs del Sprint 2
# Objetivo: exportar datasets procesados y documentación de features

processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df_train_processed.to_csv(processed_dir / "train_processed.csv", index=False)
df_test_processed.to_csv(processed_dir / "test_processed.csv", index=False)
df_train_processed.to_csv(processed_dir / "train_balanced.csv", index=False)
feature_docs.to_csv(processed_dir / "feature_docs_pipeline.csv", index=False)

print("Archivos guardados correctamente:")
print("-", processed_dir / "train_processed.csv")
print("-", processed_dir / "test_processed.csv")
print("-", processed_dir / "train_balanced.csv")
print("-", processed_dir / "feature_docs_pipeline.csv")

Archivos guardados correctamente:
- C:\Proyecto PUCP\dp261-g4\data\processed\train_processed.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\test_processed.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\train_balanced.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\feature_docs_pipeline.csv


In [31]:
# Celda 11 - PB-07 / PB-08 / PB-09: persistencia del pipeline
# Objetivo: guardar el pipeline para reutilizarlo en los siguientes sprints

models_dir = ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline_preprocessing, models_dir / "preprocessing_pipeline.pkl")

print("Pipeline guardado correctamente en:")
print("-", models_dir / "preprocessing_pipeline.pkl")

Pipeline guardado correctamente en:
- C:\Proyecto PUCP\dp261-g4\models\preprocessing_pipeline.pkl


In [32]:
# Celda 12 - PB-07 / PB-08 / PB-09: validación final y resumen
# Objetivo: comprobar que el artefacto se carga correctamente y resumir outputs

pipeline_loaded = joblib.load(models_dir / "preprocessing_pipeline.pkl")

print("Pipeline cargado correctamente.")
print("\nResumen de artefactos generados:")
print("-", processed_dir / "train_processed.csv")
print("-", processed_dir / "test_processed.csv")
print("-", processed_dir / "train_balanced.csv")
print("-", processed_dir / "feature_docs_pipeline.csv")
print("-", models_dir / "preprocessing_pipeline.pkl")

Pipeline cargado correctamente.

Resumen de artefactos generados:
- C:\Proyecto PUCP\dp261-g4\data\processed\train_processed.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\test_processed.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\train_balanced.csv
- C:\Proyecto PUCP\dp261-g4\data\processed\feature_docs_pipeline.csv
- C:\Proyecto PUCP\dp261-g4\models\preprocessing_pipeline.pkl
